#  Lab 3 — Demo Formatore

### UF-16 · ML & AI · Modulo 1 · Hands-on Lab — Regressione

> **Versione:** demo / instructor &nbsp;·&nbsp; **Durata:** ~90 min &nbsp;·&nbsp; **Ambiente:** Google Colab o Jupyter locale

Soluzioni complete di tutti i task del Lab 3: dal primo Hello World su regressione al workflow end-to-end sul dataset Boston Housing.

**Come usarlo in aula:**
- Mostralo solo dopo che i partecipanti hanno provato in autonomia.
- Le note  segnalano i punti dove tipicamente si bloccano.
- Le note  segnalano errori frequenti e punti di attenzione.

---


## 1⃣ Esercizio 1 — Hello World su regressione

>  **Messaggio chiave:** la regressione predice un **numero continuo**, non una categoria. L'API scikit-learn è **identica** alla classificazione: `fit / predict / score`. Cambia solo l'Estimator e la metrica di valutazione.


### Task 1.1 — Verifica librerie

In [ ]:
import sklearn
import numpy as np
print("scikit-learn:", sklearn.__version__)

from sklearn import datasets, model_selection, linear_model, metrics
print("Moduli importati correttamente ")

### Task 1.2 — Tre Estimator di regressione: stessa API

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

for model in [LinearRegression(), DecisionTreeRegressor(), KNeighborsRegressor()]:
    print(type(model).__name__, "→", len(model.get_params()), "iperparametri")
    print("  params:", list(model.get_params().keys()))
    print()

>  Sottolinea: tutti e tre hanno `.fit()`, `.predict()`, `.score()`. **Imparato uno, imparati tutti.**
> La differenza rispetto alla classificazione: l'output è un **numero reale**, non una classe.


### Task 1.3 — Dataset giocattolo: prezzo appartamenti

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X_train = np.array([[50, 2], [80, 3], [100, 4], [60, 2], [120, 5], [90, 3]])
y_train = np.array([150, 220, 290, 170, 350, 250])   # prezzi in k€

reg = LinearRegression()
reg.fit(X_train, y_train)

nuovo = np.array([[75, 3]])
print(f"Prezzo stimato per 75m², 3 stanze: {reg.predict(nuovo)[0]:.1f} k€")
print(f"Coefficienti (m², stanze): {reg.coef_.round(2)}")
print(f"Intercetta: {reg.intercept_:.2f}")
print()
print("Interpretazione: ogni m² aggiunge", round(reg.coef_[0], 2), "k€")
print("                 ogni stanza aggiunge", round(reg.coef_[1], 2), "k€")

>  **Lezione chiave:** i coefficienti della regressione lineare sono **interpretabili**. Ogni unità di feature aggiunge `coeff` al valore predetto.
>  **Punto di attenzione:** i coefficienti dipendono dalla scala delle feature — non sono direttamente confrontabili se le feature hanno range diversi.


---
## 2⃣ Esercizio 2 — Regressione sul dataset Boston Housing

>  Il Boston Housing dataset contiene dati su **506 quartieri** di Boston (anni '70).
> **Target (MEDV):** valore mediano delle abitazioni in migliaia di dollari — variabile **continua**.
>  **Nota etica:** il dataset contiene una variabile (`B`) legata alla composizione razziale. Non la usiamo come feature predittiva.


### Task 2.1 — Caricare ed esplorare il dataset

In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

boston = fetch_openml(name="boston", version=1, as_frame=True, parser="auto")

df = boston.frame.copy()
df.columns = [c.upper() for c in df.columns]

print("Shape:", df.shape)
print()
print(df.describe().round(2))

In [ ]:
# Valori mancanti?
print("Valori mancanti per colonna:")
print(df.isnull().sum())
print()
print(f"MEDV — min: {df['MEDV'].min():.1f}  max: {df['MEDV'].max():.1f}  media: {df['MEDV'].mean():.1f} k$")

>  **Punti di attenzione da sottolineare:**
> - `MEDV` va da 5.0 a **50.0 k$** — il valore 50 è un **troncamento artificiale** (molte case hanno esattamente 50.0). Questo è un data quality issue reale.
> - Nessun valore mancante → dataset pulito per questo lab.
> - Shape (506, 14): 13 feature + 1 target.


### Task 2.2 — Visualizzare i dati

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Distribuzione del target
axes[0].hist(df["MEDV"], bins=30, edgecolor="white", color="steelblue")
axes[0].set(title="Distribuzione MEDV", xlabel="Valore mediano (k$)", ylabel="Conteggio")
axes[0].axvline(50, color="red", linestyle="--", label="Troncamento a 50k$")
axes[0].legend()

# Heatmap correlazioni con MEDV
corr = df.corr(numeric_only=True)[["MEDV"]].sort_values("MEDV")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title("Correlazione con MEDV")

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot delle due feature più correlate
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, feat, color in zip(axes, ["RM", "LSTAT"], ["steelblue", "darkorange"]):
    ax.scatter(df[feat], df["MEDV"], alpha=0.4, s=15, color=color)
    ax.set(xlabel=feat, ylabel="MEDV (k$)", title=f"{feat} vs MEDV")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

>  **Discussione:**
> - `RM` (numero stanze) ha correlazione **positiva** ~0.70: più stanze → valore più alto. Relazione quasi lineare.
> - `LSTAT` (% basso status) ha correlazione **negativa** ~-0.74: relazione **curvilinea** → la regressione lineare farà fatica qui.
> - Il **picco a 50k$** nella distribuzione è il troncamento artificiale — un modello lineare non può catturarlo.
>  **Lezione chiave:** guardare i dati prima di modellare rivela problemi che nessun algoritmo può risolvere da solo.


### Task 2.3 — Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

# Escludi B per ragioni etiche
FEATURES = ["CRIM","ZN","INDUS","CHAS","NOX","RM","AGE","DIS","RAD","TAX","PTRATIO","LSTAT"]
X = df[FEATURES].astype(float).values
y = df["MEDV"].astype(float).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"Media MEDV — train: {y_train.mean():.2f}  test: {y_test.mean():.2f}")

>  **Punti di attenzione:**
> - Per la regressione **non usiamo `stratify`** (non ci sono classi), ma fissiamo sempre `random_state`.
> - Verifica che le medie del target nei due split siano simili: se fossero molto diverse, lo split sarebbe sbilanciato.
> - **Mai** usare il test set per scegliere feature o iperparametri.


### Task 2.4 — Addestrare la regressione lineare

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd

lr = LinearRegression()
lr.fit(X_train, y_train)

print(f"R² sul TRAINING: {lr.score(X_train, y_train):.4f}")
print()

# Coefficienti interpretabili
coef_df = pd.DataFrame({"feature": FEATURES, "coeff": lr.coef_}).sort_values("coeff")
print("Coefficienti (ordinati):")
print(coef_df.to_string(index=False))
print(f"\nIntercetta: {lr.intercept_:.4f}")

>  **Discussione:**
> - R² training ≈ 0.74–0.76: il modello spiega ~75% della varianza. Non male, ma non perfetto.
> - `RM` ha il coefficiente più alto positivo: ogni stanza aggiunge ~X k$ al valore.
> - `LSTAT` ha coefficiente negativo: più basso status → valore più basso.
>  **Punto di attenzione:** R² sul training **non misura la generalizzazione**. Serve il test set.


### Task 2.5 — Predizioni e analisi dei residui

In [ ]:
y_pred = lr.predict(X_test)

# Confronto prime 10 predizioni
comparison = pd.DataFrame({
    "Reale": y_test[:10].round(1),
    "Predetto": y_pred[:10].round(1)
})
comparison["Errore"] = (comparison["Predetto"] - comparison["Reale"]).round(1)
print(comparison.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Reale vs Predetto
axes[0].scatter(y_test, y_pred, alpha=0.5, s=20, color="steelblue")
axes[0].plot([5, 50], [5, 50], "r--", lw=1.5, label="Predizione perfetta")
axes[0].set(xlabel="Valore reale (k$)", ylabel="Valore predetto (k$)",
            title="Reale vs Predetto — LinearRegression")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribuzione residui
residui = y_pred - y_test
axes[1].hist(residui, bins=25, edgecolor="white", color="darkorange")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set(xlabel="Residuo (k$)", title="Distribuzione dei residui")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residuo medio: {residui.mean():.3f} k$  (ideale: 0)")
print(f"Std residui:   {residui.std():.3f} k$")

>  **Punti di attenzione da sottolineare:**
> 1. **Grafico Reale vs Predetto:** i punti dovrebbero stare sulla diagonale rossa. La dispersione aumenta per valori alti → il modello è meno preciso sui valori estremi.
> 2. **Troncamento a 50k$:** si vede chiaramente nel grafico — molti punti reali a 50 con predizioni diverse. Il modello non può catturare questo artefatto.
> 3. **Residui centrati sullo zero:** buon segno. Se fossero sistematicamente positivi o negativi, il modello avrebbe un **bias**.
>  **Lezione chiave:** analizzare i residui rivela **dove** sbaglia il modello, non solo quanto sbaglia.


### Task 2.6 — Valutare il modello: MAE, MSE, RMSE, R²

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def valuta_modello(y_true, y_pred, nome="Modello"):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    print(f"── {nome} ──")
    print(f"  MAE  : {mae:.2f} k$")
    print(f"  MSE  : {mse:.2f} k$²")
    print(f"  RMSE : {rmse:.2f} k$")
    print(f"  R²   : {r2:.4f}")
    return mae, rmse, r2

print("=== TEST SET ===")
valuta_modello(y_test, y_pred, "LinearRegression")
print()
print("=== TRAINING SET (confronto) ===")
valuta_modello(y_train, lr.predict(X_train), "LinearRegression")

>  **Glossario metriche:**
> | Metrica | Formula | Interpretazione |
> |---------|---------|-----------------|
> | **MAE** | `mean(|y - ŷ|)` | Errore medio in k$ — facile da spiegare |
> | **MSE** | `mean((y - ŷ)²)` | Penalizza errori grandi — sensibile agli outlier |
> | **RMSE** | `√MSE` | Stessa unità del target — più interpretabile di MSE |
> | **R²** | `1 - SS_res/SS_tot` | Frazione di varianza spiegata — 1.0 = perfetto |
>
>  **Punti di attenzione:**
> - Le metriche sul **training** sono leggermente migliori di quelle sul **test** → normale.
> - Se la differenza fosse enorme → **overfitting**.
> - **MAE** è la metrica più facile da comunicare a un non-tecnico: "in media sbagliamo di X k$".
> - Usa **sempre almeno due metriche** per avere un quadro completo.


### Task 2.7 — Sperimentare con iperparametri e confrontare modelli

**Parte A — Ridge e Lasso: regolarizzazione**

In [ ]:
from sklearn.linear_model import Ridge, Lasso
import pandas as pd

results = []
for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    for ModelClass in [Ridge, Lasso]:
        m = ModelClass(alpha=alpha)
        m.fit(X_train, y_train)
        y_p = m.predict(X_test)
        results.append({
            "Modello": type(m).__name__,
            "alpha": alpha,
            "R² train": round(m.score(X_train, y_train), 4),
            "R² test":  round(m.score(X_test, y_test), 4),
            "RMSE test": round(np.sqrt(mean_squared_error(y_test, y_p)), 3),
        })

print(pd.DataFrame(results).to_string(index=False))

**Parte B — Decision Tree Regressor: profondità dell'albero**

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_results = []
for depth in [2, 3, 5, 7, 10, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    y_p = m.predict(X_test)
    tree_results.append({
        "max_depth": str(depth),
        "R² train": round(m.score(X_train, y_train), 4),
        "R² test":  round(m.score(X_test, y_test), 4),
        "RMSE test": round(np.sqrt(mean_squared_error(y_test, y_p)), 3),
    })

print(pd.DataFrame(tree_results).to_string(index=False))

**Parte C — Grafico bias/varianza per Decision Tree**

In [ ]:
import matplotlib.pyplot as plt

depths = [2, 3, 5, 7, 10, 15]
train_r2, test_r2 = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_r2.append(m.score(X_train, y_train))
    test_r2.append(m.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_r2, "o-", label="R² training", color="steelblue")
plt.plot(depths, test_r2,  "s-", label="R² test",     color="darkorange")
plt.xlabel("max_depth")
plt.ylabel("R²")
plt.title("Decision Tree Regressor — effetto di max_depth")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Baseline: LinearRegression
lr_r2_test = lr.score(X_test, y_test)
print(f"Baseline LinearRegression R² test: {lr_r2_test:.4f}")
print(f"Miglior Decision Tree R² test:     {max(test_r2):.4f}")

>  **Lezione chiave (importante!):**
>
> **Ridge/Lasso:**
> - `alpha` piccolo → si comporta come LinearRegression
> - `alpha` grande → coefficienti si avvicinano a zero (underfitting)
> - Lasso può azzerare completamente alcuni coefficienti → **selezione automatica delle feature**
>
> **Decision Tree:**
> - `max_depth=None` → R² training ≈ 1.0 → **overfitting evidente**
> - `max_depth=3-5` → spesso supera la regressione lineare sul test
> - Il grafico mostra chiaramente il **trade-off bias/varianza**
>
>  **Punto di attenzione critico:** scegliere `max_depth` guardando il test set è **scorretto** — invalida la valutazione finale. Il modo corretto è la **cross-validation** sul training set (prossimi moduli).
>
>  La regressione lineare è un ottimo **baseline**: se un modello più complesso non la batte sul test, non vale la complessità aggiuntiva.


---
##  Chiusura del lab

In ~90 minuti abbiamo costruito il nostro **primo modello di regressione completo**, end-to-end.

| Step | scikit-learn API | Cosa abbiamo imparato |
|------|-----------------|----------------------|
| Carica i dati | `fetch_openml("boston")` | Dataset reale: 506 quartieri, 12 feature, target continuo |
| Esplora | `df.describe()` + `df.corr()` + scatter | Correlazioni, outlier, relazioni non lineari, data quality |
| Splitta | `train_test_split(test_size=0.20)` | Test set come misura onesta di generalizzazione |
| Addestra | `LinearRegression().fit(X_train, y_train)` | API uniforme — stessa di classificazione |
| Predici | `model.predict(X_test)` | Output numerico continuo, non una classe |
| Analizza residui | Scatter reale vs predetto + istogramma | Bias sistematici e artefatti nei dati |
| Valuta | `MAE`, `MSE`, `RMSE`, `R²` | Quattro metriche complementari |
| Tuning | Ridge/Lasso (`alpha`) + Decision Tree (`max_depth`) | Trade-off bias/varianza, overfitting/underfitting |

### Domande aperte da fare alla classe

> - "Quale metrica usereste per comunicare le prestazioni a un cliente immobiliare? MAE o R²?"
> - "Il troncamento a 50k$ è un problema del modello o dei dati? Come lo risolvereste?"
> - "Se il RMSE sul training fosse 1.0 k$ e sul test 8.0 k$, cosa significherebbe?"

 **Prossimo modulo:** cross-validation, GridSearchCV, Pipeline, e altri algoritmi di regressione.
